In [ ]:

%autoreload 2

In [27]:
%reload_ext autoreload
import os, sys, random
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

import matplotlib.pyplot as plt
from matplotlib import gridspec, rcParams
from fish import Gafftopsail
sys.path.append(r'/Users/zichenhe/miniforge3/envs/naumann_lab/2ptank/')#(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils, barcode
path = ('C://Data//Imaging//260425_overlap/fish11/')
rcParams['font.size'] = 10

In [ ]:
fish = Gafftopsail(path, filelist = ['stimulus', 'imaging'], sequence = 5)
fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]

__EACH STIM__

In [ ]:
#axis 0: trial; axis 1: neuron; axis 2: frame
#from stationary start to duration + 20
f_pertrial_dict = barcode.get_pertrial_f(fish)

In [ ]:
stim_list = ['dot_l', 'dot_r', 'forward', str(['dot_l', 'forward']), str(['dot_r', 'forward']),
            'left', str(['dot_l', 'left']), str(['dot_r', 'left']),
             'right', str(['dot_l', 'right']), str(['dot_r', 'right']),
             'backward', str(['dot_l', 'backward']), str(['dot_r', 'backward'])]

stim_dict = {'dot_l': 'darkgray', 'dot_r': 'black', 
             'forward': 'darkgreen', str(['dot_l', 'forward']): 'lightgreen', str(['dot_r', 'forward']): 'limegreen',
            'left': 'mediumblue', str(['dot_l', 'left']):'lightsteelblue', str(['dot_r', 'left']):'royalblue',
             'right': 'maroon', str(['dot_l', 'right']): 'lightcoral', str(['dot_r', 'right']): 'indianred',
             'backward': 'indigo', str(['dot_l', 'backward']): 'plum', str(['dot_r', 'backward']):'mediumpurple' }

In [ ]:
def get_response_index_f(fish, f_pertrial_dict, stim_list, baseline_s =10, response_s = 10):
    f_response_dict = {}
    for stim in stim_list:
        stim_df = fish.stimulus_df[fish.stimulus_df['stim_name'] == stim].reset_index(drop=True)
        stationary_time = stim_df['stationary_time'].iloc[0]
        if '[' in stim:#overlap
            stationary_time = min(stationary_time)
        #define plotting range (what are the frames of data to grab in general)
        
        stationary_frame = int(stationary_time/fish.image_s)
        baseline_frame = int((stationary_time - baseline_s)/ fish.image_s)#how many frame before stationary frame
        response_frame = int((stationary_time + response_s)/ fish.image_s)#how many frame after stationary frame
        #get all the baseline per trial
        f_pertrial = f_pertrial_dict[stim]
        f_pertrial_baseline = f_pertrial[:, :, baseline_frame:stationary_frame]
        f_pertrial_response = f_pertrial[:, :, stationary_frame:response_frame]
        f_pertrial_baselinemean = np.mean(f_pertrial_baseline, axis = 2)
        f_pertrial_responsemax = np.mean(f_pertrial_response, axis = 2)
        f_pertrial_response = np.subtract(f_pertrial_responsemax, f_pertrial_baselinemean)
        f_response_dict[stim] = f_pertrial_response.mean(axis = 0)
    return f_response_dict

In [ ]:
baseline_s = 20
response_s = 20
f_response_dict = get_response_index_f(fish, f_pertrial_dict, stim_list, baseline_s =baseline_s, response_s = response_s)#trialxneuronxframe

__get visually responsive cells__

In [ ]:
vis_neurons = barcode.select_visbarcode(fish, f_pertrial_dict, baseline_s = 5, response_s =20, perc_trial_threshold = 0.8)

__neuron pop dynamics__

In [ ]:
f_pertrial_matrix = np.concatenate([f_pertrial_dict[s][:, vis_neurons] for s in stim_list], axis = 0)#trial, stim (6 x 10 + 8 *5) x  neuron x frames
trial_number_acc = {s: f_pertrial_dict[s].shape[0] for s in stim_list}
f_pertrial_matrix = f_pertrial_matrix.transpose(0, 2, 1).reshape(-1, f_pertrial_matrix.shape[1])
f_pertrial_matrix = np.nan_to_num(f_pertrial_matrix)
f_pertrial_matrix.shape#goal: stim, trial, frame xneurons

In [ ]:
components = 20
pca = PCA(n_components=components)   # or None to keep all
X_pca = pca.fit_transform(f_pertrial_matrix)#stim, trial, frame x dimension


In [ ]:
def label_stimuli(f_pertrial_dict, trial_number_acc, X_pca):
    #label each frame
    stimulus_type = []
    stimulus = []
    behaviors = []
    actual_stimulus = []
    trial_idx = []
    intrial_frame = []
    frame_num = f_pertrial_dict[list(f_pertrial_dict.keys())[0]].shape[-1]
    for stim in trial_number_acc.keys():
        stim_df = fish.stimulus_df[fish.stimulus_df.stim_name == stim].reset_index(drop=True).iloc[0]
        behavior = list(fish.stimulus_df[fish.stimulus_df.stim_name == stim].hunting_eye)
        stationary_time = stim_df.stationary_time
        if '[' in stim:
            grating_stationary_time = stationary_time[1]
            dot_stationary_time = stationary_time[0] +5
            dot_stationary_frame = int(dot_stationary_time/fish.image_s)
            grating_stationary_frame = int(grating_stationary_time/fish.image_s)
        elif 'dot' in stim:
            dot_stationary_time = stationary_time + 5
            grating_stationary_frame = np.nan
            dot_stationary_frame = int(dot_stationary_time/fish.image_s)
        else:
            dot_stationary_frame = np.nan
            grating_stationary_time = stationary_time
            grating_stationary_frame = int(grating_stationary_time/fish.image_s)
        for trial in range(trial_number_acc[stim]):
            stimulus.extend([stim] * frame_num)
            actual_stimulus.extend(['stationary'] * int(np.nanmin([dot_stationary_frame, grating_stationary_frame])))
            behaviors.extend(np.repeat(behavior[trial], frame_num))
            if '[' in stim:#overlap
                actual_stimulus.extend([stim.split("'")[3]] * (dot_stationary_frame - grating_stationary_frame))
                actual_stimulus.extend([stim] * (frame_num - dot_stationary_frame))
                stimulus_type.extend(['overlap'] * frame_num)
            elif 'dot' in stim: #dot stim pure
                actual_stimulus.extend([stim] * (frame_num - dot_stationary_frame))
                stimulus_type.extend(['dot'] * frame_num)
            else:#pure grating
                actual_stimulus.extend([stim] * (frame_num - grating_stationary_frame))
                stimulus_type.extend(['grating'] * frame_num)
            trial_idx.extend([trial] * frame_num)
            intrial_frame.extend(list(range(frame_num)))

    X_pca = pd.DataFrame(X_pca)
    X_pca.loc[:, 'stimulus'] = stimulus
    X_pca.loc[:, 'trial'] = trial_idx
    X_pca.loc[:, 'actual_stimulus'] = actual_stimulus
    X_pca.loc[:, 'intrial_frame'] = intrial_frame
    X_pca.loc[:, 'behavior'] = behaviors
    return X_pca
X_pca = label_stimuli(f_pertrial_dict, trial_number_acc, X_pca)

In [ ]:
#ge average traces
X_pca_trialmean = X_pca.groupby(by = ['stimulus', 'actual_stimulus','intrial_frame']).mean()#we can group by behavior later
X_pca_trialmean = X_pca_trialmean.drop(['trial', 'behavior'], axis = 1).reset_index(drop = False)
X_pca_trialmean.sort_values(by = 'intrial_frame', inplace = True)

In [ ]:
def plot_components(components, X_pca_trialmean):
    fig, ax = plt.subplots(components, 5, figsize = (10, 20))
    interesting_stims = [s for s in stim_list if '[' not in s]
    interesting_stims.extend(['stationary'])
    grating_stim = [s for s in interesting_stims if 'dot' not in s and 'stationary' not in s]
    dot_stim = [s for s in interesting_stims if 'dot' in s]
    overlap_stim = [s for s in stim_list if '[' in s]

    for component in range(components):
        ax_distribution = ax[component, 0]
        sns.kdeplot(X_pca_trialmean, y = component, hue = 'actual_stimulus', hue_order = interesting_stims, palette = [utils.stim_colors[s] for s in interesting_stims], legend = False,  common_norm = False, ax = ax_distribution)
        ax_distribution.set_title(component)

        ax_acrosstime_gratings = ax[component, 1]
        for m, linestyle in zip([True, False], ['-', ':']):
            for g_stim in grating_stim:
                X_stim = X_pca_trialmean[(X_pca_trialmean.stimulus == g_stim) & (X_pca_trialmean.behavior == m)]
                ax_acrosstime_gratings.plot(X_stim.intrial_frame, X_stim.loc[:, component], color = utils.stim_colors[g_stim], linestyle = linestyle)

            ax_acrosstime_dot = ax[component, 2]
            for d_stim in dot_stim:
                X_stim = X_pca_trialmean[(X_pca_trialmean.stimulus == d_stim) & (X_pca_trialmean.behavior == m)]
                ax_acrosstime_dot.plot(X_stim.intrial_frame, X_stim.loc[:, component], color = utils.stim_colors[d_stim], linestyle = linestyle)

            ax_acrosstime_overlap = ax[component, 3]
            for o_stim in overlap_stim:
                X_stim = X_pca_trialmean[(X_pca_trialmean.stimulus == o_stim) & (X_pca_trialmean.behavior == m)]
                ax_acrosstime_overlap.plot(X_stim.intrial_frame, X_stim.loc[:, component], color = utils.stim_colors[o_stim.split("'")[3]], linestyle = linestyle)
            for axis in ax[component]:
                axis.set_ylim(np.percentile(X_pca_trialmean.loc[:, component], 0.0001), np.percentile(X_pca_trialmean.loc[:, component], 99.999))
    return fig
plot_components(components, X_pca_trialmean)
plt.show()
plt.close()

In [28]:
#find interesting components that differentiate graitngs and dot consistently
for g, d in itertools.product(['forward', 'left', 'right', 'backward'], ['dot_l', 'dot_r']):
    X_g = X_pca_trialmean[X_pca_trialmean.stimulus == g].drop(['stimulus', 'actual_stimulus', 'intrial_frame'], axis = 1)
    X_d = X_pca_trialmean[X_pca_trialmean.stimulus == d].drop(['stimulus', 'actual_stimulus', 'intrial_frame'], axis = 1)
    #for each component, calculate the cumulative differentiation of these two stimuli across frame
    diffs = np.subtract(X_g, X_d)


In [ ]:
interesting_component = 19
neuron_contribution = pca.components_[interesting_component]
fig, ax = plt.subplots(1, 1)
ax.imshow(fish.img_dict[3], cmap = 'gray', origin = 'lower')
vmax = max(np.abs(np.nanpercentile(neuron_contribution, 95)), np.abs(np.nanpercentile(neuron_contribution, 5)))
ax.scatter(fish.pos_all.loc[vis_neurons, 'xpos'], fish.pos_all.loc[vis_neurons, 'ypos'], s  =.5, alpha = 0.8, c=neuron_contribution, cmap = 'bwr', vmax = vmax, vmin = -vmax)
plt.show()
plt.close()

In [29]:
X_g

,stimulus,actual_stimulus,intrial_frame,0,1,2,3,4,5,6,...,10,11,12,13,14,15,16,17,18,19
976,backward,stationary,0,2.190603,0.851051,0.552303,-0.952992,-1.249110,-0.181827,-0.359052,...,-0.023666,-0.380013,0.084910,0.427617,0.181114,0.094391,-0.472607,0.219273,0.091127,-0.004889
977,backward,stationary,1,2.209169,0.885055,0.578883,-0.908510,-1.269633,-0.180434,-0.365092,...,0.008546,-0.377617,0.087436,0.460492,0.163926,0.177315,-0.491880,-0.049196,0.021893,-0.061576
978,backward,stationary,2,2.165959,0.913150,0.598316,-0.863930,-1.269630,-0.212478,-0.381951,...,0.048798,-0.390247,0.068479,0.462120,0.136046,0.247802,-0.371101,-0.211434,0.174734,-0.090939
979,backward,stationary,3,2.053283,0.860304,0.612066,-0.823936,-1.223648,-0.261291,-0.387003,...,0.091195,-0.439620,0.049682,0.436270,0.142234,0.237895,-0.198735,-0.120949,0.381186,-0.068858
980,backward,stationary,4,1.826709,0.728928,0.603835,-0.779893,-1.129603,-0.313431,-0.360983,...,0.117335,-0.486985,0.040975,0.406432,0.171239,0.168260,-0.106088,0.135152,0.406187,-0.025415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
971,backward,backward,108,-2.399366,-0.654773,-0.757904,-0.181488,0.413568,-0.172536,0.130489,...,0.472373,0.300203,-0.016389,0.019483,0.100534,-0.265633,-0.033399,0.171285,-0.036292,0.049302
972,backward,backward,109,-2.383283,-0.618468,-0.720146,-0.163813,0.402111,-0.187555,0.139054,...,0.464869,0.308406,-0.022542,0.045254,0.078788,-0.218206,-0.132873,-0.059337,-0.178060,0.033410
973,backward,backward,110,-2.366435,-0.600793,-0.666817,-0.162999,0.377925,-0.204727,0.154862,...,0.464127,0.296733,-0.024898,0.057838,0.046088,-0.169883,-0.115459,-0.313746,-0.099312,0.022037
974,backward,backward,111,-2.369305,-0.592745,-0.620955,-0.171580,0.364541,-0.220316,0.172876,...,0.451012,0.255065,-0.032218,0.057392,0.035866,-0.176834,-0.018852,-0.361257,0.105089,0.028373


In [30]:
components

20